Сегодня не чистим таблицу и не рисуем гистограммы. Сегодня учим программу **угадывать цену машины** по нескольким числам: год, пробег, объём двигателя, километры в год.

Мы показываем много примеров «вот признаки — вот цена». Программа запоминает правило. Потом даём новые признаки без цены — она выдаёт число. Правило может ошибаться. Вся пара про то, как это правило получить и как понять, насколько оно врёт.



## 1. Что именно мы просим модель сделать

Представьте риелтора, который никогда не видел машину вживую. Ему дают бумажку: 2015 год, 120 тысяч км, двигатель 1600. Он называет цену. Иногда попадает, иногда нет. Наша модель — такой риелтор, только считает по формуле, а не «на глаз».

Слова, которые дальше будут везде.

**Объект** — одна строка таблицы, одно объявление.

**Признаки** — то, что модели можно показать, когда цены ещё нет. Год, пробег, объём, пробег в год. Это вход.

**Целевая переменная** — то, что хотим получить. У нас цена `Price`. Это выход. Задача называется **регрессия**: ответ — число, не «да/нет» и не «красная/синяя».

**Обучение** — настройка правила по примерам, где цена уже известна.

**Предсказание** — применение правила к машине, цену которой мы как будто не знаем.

Модель не «понимает», что машина битая или что продавец завысил цену. Она видит только столбцы, которые мы ей отдали.

**Остановитесь:** какая колонка здесь является ответом? Можно ли передать её модели как входной признак, когда цену мы ещё не знаем?


### Подключаем библиотеки

Библиотека называется **scikit-learn**. Ставится командой `pip install scikit-learn`, а в коде пишется `sklearn`. Это как pandas: имя пакета на сайте и имя в `import` чуть отличаются.

`from sklearn... import ...` — взять один инструмент из большого ящика, не тащить весь ящик.

Установка при необходимости: `pip install pandas numpy matplotlib scikit-learn joblib`. После установки перезапустите ядро.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
import sklearn

print('Версия scikit-learn:', sklearn.__version__)


Появился номер версии. Это проверка, что библиотека живая. Модель ещё не создана. Модель появится, когда напишем `fit`.

**Остановитесь:** чем подключение библиотеки отличается от обучения модели?


### Загружаем данные

Берём уже обработанный файл прошлой практики, `cars_processed.csv`. Не сырой `cars.csv` и не таблицу средних по группам. Если файла рядом с ноутбуком нет, код качает пример из интернета.


In [ ]:
data_path = Path('cars_processed.csv')
if data_path.exists():
    df = pd.read_csv(data_path)
else:
    df = pd.read_csv(
        'https://raw.githubusercontent.com/MVRonkin/BasicDataAnalysisCourse/'
        'refs/heads/main/Workshops/cars_processed.csv'
    )
df = df.rename(columns={'Price(euro)': 'Price', 'Engine_capacity(cm3)': 'Engine'})
df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed:')])
df = df.reset_index(drop=True)
NUMERIC_FEATURES = ['Year', 'Distance', 'Engine', 'km_year']
if df[NUMERIC_FEATURES + ['Price']].isna().any().any():
    raise ValueError('В выбранных колонках есть пропуски: вернитесь к подготовке данных.')
if not np.isfinite(df[NUMERIC_FEATURES + ['Price']].to_numpy()).all():
    raise ValueError('В выбранных колонках есть бесконечные значения.')
if not (df['Price'] > 0).all():
    raise ValueError('Для этого учебного примера нужны положительные цены.')
df.head(3)


Что делает ячейка по шагам.

1. Ищет файл и читает таблицу.
2. Переименовывает длинные имена в короткие `Price` и `Engine`, чтобы не мучиться со скобками.
3. Выкидывает служебный столбец `Unnamed`, если Excel или сохранение его добавили.
4. Нумерацию строк сбрасывает с нуля.

`NUMERIC_FEATURES` — список из четырёх входов. Дальше везде этот список, чтобы не опечататься в имени столбца.

Три проверки — предохранитель. Пустая клетка, бесконечность или цена ноль/минус — для этой учебной модели нельзя. Если ячейка красная, чистка данных ещё не закончена, модель не начинаем.


## 2. X — вопросы модели, y — известные ответы

Sklearn ждёт данные в двух коробках, не в одной таблице.

`X` — вопросы. Несколько столбцов, **без цены**.

`y` — ответы. Один столбец цен, по одному числу на строку.

Как контрольная: слева условия задач, справа правильные ответы. Номер задачи слева и справа должен быть один и тот же.


In [ ]:
X = df[NUMERIC_FEATURES].copy()
y = df['Price'].copy()

display(X.head(3))
display(y.head(3))


`.copy()` — своя копия, чтобы случайно не портить исходный `df`. На экране сначала четыре числа, потом отдельно цена.

Цена в `X` не должна появиться. Иначе модель «списывает» ответ с листа.

**Остановитесь:** где находится цена автомобиля с индексом 0?


### Проверяем размеры

Для `X` форма `(n, p)` означает n объектов и p признаков. Для `y` форма `(n,)` означает n ответов.


In [ ]:
print('X:', X.shape)
print('y:', y.shape)
print('Индексы совпадают:', X.index.equals(y.index))


Первые числа у `X` и `y` должны совпасть. Если у `X` 30 тысяч строк, а у `y` 29 — Python, может, и посчитает, но это уже чужие ответы к чужим машинам.

Индексы — номера строк Pandas. Они должны быть одинаковые у `X` и `y`. Иначе «машина 7» получит цену от «машины 15».


**Мини-задание.** Покажите в одной маленькой таблице признаки и цену первых трёх объектов. Объясните, почему такую таблицу удобно смотреть глазами, но целиком нельзя передавать как `X`.


In [ ]:
# Мини-задание: показываем признаки и цену первых трёх объектов в одной таблице
display(df[NUMERIC_FEATURES + ['Price']].head(3))

# Объяснение:
# Такая таблица удобна для визуального контроля и проверки корректности данных.
# Однако целиком передавать её в X нельзя: наличие столбца 'Price' приведет к 'утечке целевой переменной'
# (data leakage), из-за чего модель просто «спишет» ответ, не выучив реальной закономерности.


## 3. Учим на одной части, проверяем на другой

Если дать модели все машины, потом спросить цену тех же машин — она может «запомнить» примеры, а не выучить правило. Как выучить билеты наизусть и растеряться на новом вопросе.

Поэтому таблицу режем на три куска. Случайно, но с зафиксированным зерном, чтобы у всех получилось одно и то же.

| Часть | Примерная доля | Для чего |
|---|---:|---|
| `train` | 60% | Учим. Здесь модели можно показывать цены |
| `valid` | 20% | Сравниваем варианты. Цены есть у нас, модели при предсказании их не даём |
| `test` | 20% | Финальный экзамен. Сегодня **метриками не трогаем** — это работа 06 |

Зачем два «экзамена»? Valid можно подглядывать: «эта модель лучше, та хуже». Если так крутить долго, valid тоже становится шпаргалкой. Test держат до последнего, чтобы не подкрутить ответ под него.

Почему тогда два вызова `train_test_split`, а не один? Функция умеет только **одну** резку: стопка → две стопки. Три кучи из одной команды она не делает. Поэтому режем так:

1. Берём **всю** таблицу. Отрезаем 40% в сторону. Остальные 60% — это уже готовый **train**. Его больше не режем.
2. Те 40%, что отложили, пока не train и не test: это просто «ещё не разобрали». Их называют `holdout`, «отложенная часть».
3. Теперь режем **только holdout**, пополам: половина станет valid, половина — test. Половина от 40% — это 20% от всей таблицы. Другую половину тоже 20%. Train в этот вызов **не передаём**.

На пальцах: пирог 10 кусков. Сначала 6 кусков на тарелку «учить», 4 — в сторонку. Потом эти 4 делим: 2 на «проверить сегодня», 2 на «не открывать». Шесть кусков с первой тарелки никто второй раз не режет.


### Первый вызов

Вход — весь `X` и весь `y`. `test_size=0.4` значит: **вторая** куча будет 40% строк, первая — оставшиеся 60%. Имена слева от `=`: первая куча — `X_train` / `y_train`, вторая — `X_holdout` / `y_holdout`. Слово `test` в аргументе здесь обманчивое: это не наш финальный test, а просто «доля второй кучи». Holdout ещё предстоит разрезать.

Функция режет `X` и `y` **вместе**: какая строка уехала в train у признаков, та же уехала у цен. `random_state=42` — не «магическое качество», а фиксация жребия. Другое число — другой набор строк.


In [ ]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.4, random_state=42
)
print('Обучение:', X_train.shape, y_train.shape)
print('Отложенная часть:', X_holdout.shape, y_holdout.shape)


### Второй вызов

Вход уже **не** вся таблица, а только holdout. `test_size=0.5` — режем эти 40% пополам. Первая половина — `X_valid` / `y_valid`, вторая — `X_test` / `y_test`. `random_state` другой (43), чтобы второй жребий не путать с первым. Train сюда **не передаём**. После этого holdout нам больше не нужен: его строки разошлись по valid и test.


In [ ]:
X_valid, X_test, y_valid, y_test = train_test_split(
    X_holdout, y_holdout, test_size=0.5, random_state=43
)
split_sizes = pd.Series({'train': len(X_train), 'valid': len(X_valid), 'test': len(X_test)})
display(split_sizes.to_frame('Число объектов'))
print('Частей в сумме:', split_sizes.sum(), 'Исходных строк:', len(X))


Проверка на пальцах: три числа `len(X_train)`, `len(X_valid)`, `len(X_test)` в сумме должны дать `len(X)`. Чуть не те проценты из-за округления — нормально: 40% от 10 тысяч — ровно, от 10 001 уже не ровно.

Это не «будущий рынок». Даты объявлений нет. Если одна и та же машина два раза в файле, она может попасть и в train, и в valid — оценка станет слишком радужной. На этой паре помним: случайный разрез строк, не священная правда.


### Проверяем отсутствие пересечения

`intersection` — общие номера строк. Хотим нули: одна машина не должна сидеть в двух частях сразу.


In [ ]:
print('train ∩ valid:', len(X_train.index.intersection(X_valid.index)))
print('train ∩ test:', len(X_train.index.intersection(X_test.index)))
print('valid ∩ test:', len(X_valid.index.intersection(X_test.index)))


Нули подтверждают разделение строк, но не доказывают, что в файле нет скрытых повторных объявлений.

**Мини-задание.** Своими словами: что нельзя делать с `y_test` во время выбора модели?


In [ ]:
# Ответ на мини-задание:
# Во время выбора и настройки модели y_test НЕЛЬЗЯ использовать для вычисления метрик,
# подбора гиперпараметров или принятия решений о том, какая модель лучше.
# y_test хранится до самого конца и используется строго один раз — для финального экзамена.


## 4. До сложной модели: один ответ для всех автомобилей

Прежде чем строить «умную» формулу, нужен пол. Самый простой прогноз, с которым потом сравнивать.

Идея: всем машинам назвать **одну и ту же** цену — медиану цен из train. Не среднюю: среднее тянется к космическим миллионам, медиана — «типичная середина». Признаки вообще не используются. Это как ответить «ну, машины обычно столько стоят» и не смотреть на год.

`DummyRegressor` — официальная простая модель-ориентир в scikit-learn. `strategy='median'` выбирает медиану.


### Создаём модель-ориентир

Первая строка импортирует класс. Вторая создаёт объект. Создание объекта ещё не использует данные.


In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy='median')
baseline


`baseline` пока не знает медианную цену. Создание и обучение — разные шаги.

**Остановитесь:** на каком следующем шаге ему станут известны обучающие ответы?


### fit: обучаем

`fit` — «посмотри на примеры и настройся». Для dummy настройка крошечная: запомнить медиану `y_train`. Признаки `X_train` тут почти для вида, чтобы интерфейс был как у всех моделей: всегда `fit(X, y)`.


In [ ]:
baseline.fit(X_train, y_train)
print('Медиана обучающих цен:', y_train.median())
print('Значение в модели:', baseline.constant_)


Оба значения должны совпасть. Подчёркивание в `constant_` — договор sklearn: так помечают числа, которые появились **после** обучения. Пока `fit` не вызван, этого атрибута нет или он пустой.

На валидации модель ещё не обучалась.


### predict: получаем оценки

`predict` — «дай оценку». На вход только признаки, **без** цен. Сколько строк подали, столько чисел получили.


In [ ]:
baseline_valid_pred = baseline.predict(X_valid)
print('Предсказаний:', baseline_valid_pred.shape)
print('Первые пять:', baseline_valid_pred[:5])


Все числа одинаковые: мы так и просили, одна медиана на всех.

Результат — массив NumPy без индекса Pandas. Первое число — про первую строку `X_valid`, не про строку 0 исходного файла.

Зачем этот шаг? Чтобы следующая модель имела с чем сравниться. Если «умная» модель врёт так же, как медиана, признаки пока не помогли.

**Остановитесь:** почему в аргументах `predict` нет цен?


## 5. Как измерить ошибку в евро

Прогноз 12 тысяч, правда 10 тысяч. Ошибка со знаком: +2000, завышение. Если наоборот — минус, занижение. Для «насколько промахнулись» берут модуль: 2000 в обе стороны одинаково плохи, если нам не важно направление.

**MAE** — среднее таких модулей по всем проверяемым машинам. Читается: «в среднем ошибаемся на столько-то евро».

Это не «каждая машина в пределах MAE». Половина может быть хуже. Это не процент. Это не «модель смещена вверх»: плюсы и минусы в MAE уже стерты.


### Сначала считаем руками на трёх автомобилях


In [ ]:
toy_true = np.array([10_000, 20_000, 30_000])
toy_pred = np.array([12_000, 17_000, 31_000])
toy_error = toy_pred - toy_true
display(pd.DataFrame({'Цена': toy_true, 'Прогноз': toy_pred,
                      'Ошибка': toy_error, 'Абсолютная ошибка': np.abs(toy_error)}))
print('Средняя ошибка с направлением:', toy_error.mean())
print('MAE:', np.abs(toy_error).mean())


Знаки могут усредниться в ноль, а по модулю ошибки живые.

**Мини-задание.** Объясните, почему нулевая средняя ошибка здесь не означает идеальные прогнозы.


In [ ]:
# Ответ на мини-задание:
# Нулевая средняя ошибка без модуля не означает идеальный прогноз, так как положительные промахи
# (переоценка цены) и отрицательные промахи (недооценка цены) взаимоуничтожаются при суммировании.
# Например, ошибки +5000 и -5000 в среднем дают 0, но фактически оба прогноза сильно ошибочны.


### Теперь считаем MAE модели-ориентира

Функция принимает известные ответы **первым** аргументом, прогнозы — вторым. Она не обучает модель.


In [ ]:
from sklearn.metrics import mean_absolute_error

baseline_valid_mae = mean_absolute_error(y_valid, baseline_valid_pred)
print(f'MAE ориентира на валидации: {baseline_valid_mae:.1f} евро')


Запишите число: это линейка для всех следующих моделей. Единицы — евро, не проценты.

Следующие модели сравниваем на **тех же** `X_valid` и `y_valid`. Иначе сравниваете разные экзамены.


## 6. Линейная регрессия сначала по одному признаку

Самая простая «не глупая» идея: цена растёт с годом по прямой.

Прогноз = свободный член + коэффициент × год.

Свободный член `b` — сдвиг. Коэффициент `w` — насколько меняется прогноз, если год увеличили на один.

Sklearn подбирает `b` и `w` так, чтобы сумма **квадратов** ошибок на train была поменьше. Не MAE, а квадраты: крупные промахи наказываются сильнее.

Год — слабый признак. Марка, состояние, комплектация сюда не входят. Это учебный шаг, не лучшая модель рынка.


### Подготовим таблицы с одним признаком

Двойные скобки `[['Year']]` оставляют **таблицу** с одним столбцом: форма `(n, 1)`. Одна скобка `['Year']` даёт Series `(n,)`. Sklearn для `fit` хочет таблицу.


In [ ]:
X_train_year = X_train[['Year']]
X_valid_year = X_valid[['Year']]
print(X_train_year.shape)
display(X_train_year.head(3))


Машины те же, что в train и valid. Ответы `y_train` / `y_valid` не пересобираем.

**Остановитесь:** чем `X_train['Year']` отличается по форме от `X_train[['Year']]`?


### Создаём объект линейной регрессии

Снова только создание. Коэффициентов ещё нет.


In [ ]:
from sklearn.linear_model import LinearRegression

year_model = LinearRegression()
year_model


### Обучаем и смотрим параметры


In [ ]:
year_model.fit(X_train_year, y_train)
print('Коэффициент w:', year_model.coef_[0])
print('Свободный член b:', year_model.intercept_)


После `fit` появились `coef_` и `intercept_`.

`w` — про изменение **прогноза**, не доказательство «каждый новый год выпуска причинно добавляет столько евро».

`b` математически соответствует году 0, которого в данных нет. Не интерпретируйте его как «цену нулевого года».


### Предсказываем и сравниваем с ориентиром


In [ ]:
year_valid_pred = year_model.predict(X_valid_year)
year_valid_mae = mean_absolute_error(y_valid, year_valid_pred)
print(f'MAE по одному году: {year_valid_mae:.1f} евро')
print(f'MAE ориентира: {baseline_valid_mae:.1f} евро')


Два числа рядом, меньше — лучше. На этом разрезе примерно так: MAE по году около **5133** евро, у dummy около **5944**. Прямая по году ошибается меньше медианы примерно на 800 евро. Значит год **не бесполезен**: модель хоть как-то отличает новые машины от старых, а не говорит всем одну цену.

Это ещё не «хорошая модель». Ошибка в пять тысяч евро на объявлении — много. Год один, без пробега, мощности и марки. Мы только доказали, что линейка сильнее самого глупого ориентира. Дальше добавим остальные признаки и посмотрим, упадёт ли MAE ещё.


### Посмотрим на одну строку

`iloc[[0]]` сохраняет таблицу из одной строки. Это первая строка **валидации**, не первая в исходном csv.


In [ ]:
one_car = X_valid_year.iloc[[0]]
one_prediction = year_model.predict(one_car)[0]
one_true_price = y_valid.iloc[0]
display(one_car)
print(f'Известная цена: {one_true_price:.1f} евро')
print(f'Прогноз: {one_prediction:.1f} евро')
print(f'Абсолютная ошибка: {abs(one_prediction - one_true_price):.1f} евро')


Модель цену не получала: мы сравнили уже после. Ошибка одной машины может быть далеко от MAE всей кучи.

**Мини-задание.** Повторите просмотр для второго и третьего автомобилей. Модель заново не обучайте.


In [ ]:
# Для 2-го автомобиля (индекс 1 в валидации):
car_2 = X_valid_year.iloc[[1]]
pred_2 = year_model.predict(car_2)[0]
true_2 = y_valid.iloc[1]
print(f'Авто 2 | Известная цена: {true_2:.1f} евро | Прогноз: {pred_2:.1f} евро | Абс. ошибка: {abs(pred_2 - true_2):.1f} евро')

# Для 3-го автомобиля (индекс 2 в валидации):
car_3 = X_valid_year.iloc[[2]]
pred_3 = year_model.predict(car_3)[0]
true_3 = y_valid.iloc[2]
print(f'Авто 3 | Известная цена: {true_3:.1f} евро | Прогноз: {pred_3:.1f} евро | Абс. ошибка: {abs(pred_3 - true_3):.1f} евро')


## 7. Четыре признака и StandardScaler

Теперь входы все четыре. Год около 2000, пробег около сотен тысяч, объём около 1500. Для человека это разные смыслы. Для части алгоритмов это ещё и разные **масштабы**: большое число «кричит» громче маленького.

Обычная линейная регрессия без штрафа на коэффициенты часто переживёт разные единицы. Масштабирование всё равно учим: в следующей работе понадобится соседям, а интерфейс тот же `fit` / `transform`.

`StandardScaler` делает так: из значения вычесть среднее **по train**, поделить на стандартное отклонение **по train**. После этого типичные значения крутятся около нуля.

Это не проценты, не отрезок от 0 до 1 и не «нормальное распределение». Число может быть −2 или +4.


### Маленький пример до полной таблицы


In [ ]:
from sklearn.preprocessing import StandardScaler

toy_scaler = StandardScaler()
toy_distance = pd.DataFrame({'Distance': [10_000, 20_000, 30_000]})
toy_scaler.fit(toy_distance)
print('Среднее:', toy_scaler.mean_)
print('Стандартное отклонение:', toy_scaler.scale_)
print('Преобразованные значения:', toy_scaler.transform(toy_distance).ravel())


Середина стала нулём.

**Остановитесь:** что получится у значения, равного обучающему среднему?


### fit для scaler только на train

Scaler **тоже учится**. Ему надо запомнить среднее и разброс. Учится только на train. Если посчитать среднее по всей таблице, valid уже «подсказал» свои числа. Это **утечка**: экзамен повлиял на шпаргалку.

`y` scaler не нужен: цены не трогаем, масштабируем входы.


In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)
display(pd.DataFrame({'Признак': NUMERIC_FEATURES,
                      'Среднее train': scaler.mean_, 'Масштаб train': scaler.scale_}))


Четыре средних и четыре масштаба — по одному на признак.


### transform для train и valid

На valid только `transform`, не новый `fit`. Иначе у двух кусков будут разные линейки.


In [ ]:
X_train_scaled = scaler.transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

display(pd.DataFrame(X_train_scaled, columns=NUMERIC_FEATURES, index=X_train.index).head(3))
print('Train:', X_train_scaled.shape, 'Valid:', X_valid_scaled.shape)


`fit_transform(X_train)` — ленивая запись «обучись и сразу преобразуй train». На valid так нельзя.

`transform` отдаёт массив NumPy. Имена столбцов стёрлись, поэтому для печати мы их вернули руками. Форма та же: столько же строк и столбцов.

**Мини-задание.** Почему нельзя отдельно обучить второй scaler на `X_valid`?


In [ ]:
# Ответ на мини-задание:
# Отдельно обучать scaler на X_valid нельзя, так как это нарушит единство линейки измерений.
# Валидационная выборка должна преобразовываться строго по среднему и масштабу, посчитанным на TRAIN.
# Обучение scaler на valid приведет к утечке информации (data leakage) из валидационного набора.


### Новая линейная модель по четырём признакам

Формула та же идея: свободный член плюс сумма «коэффициент × признак», только признаков четыре.


In [ ]:
numeric_model = LinearRegression()
numeric_model.fit(X_train_scaled, y_train)
numeric_valid_pred = numeric_model.predict(X_valid_scaled)
numeric_train_pred = numeric_model.predict(X_train_scaled)
print('Число коэффициентов:', len(numeric_model.coef_))


Четыре коэффициента — по одному на столбец. Цену не масштабировали, прогноз в евро.

На `predict` надо подавать тот же вид, на котором учили: здесь `X_valid_scaled`. Если подать сырой `X_valid`, единицы другие и прогноз поедет.


## 8. Сравнение, таблица ошибок и график

MAE мы уже знаем. Рядом часто ставят **RMSE**: сначала квадрат ошибки, среднее, корень. Снова евро, но крупные промахи бьют больнее.

MAE ближе к «типичному промаху по модулю». RMSE злее к хвостам.

Считаем на train и на valid отдельно. Смешивать куски нельзя.


In [ ]:
from sklearn.metrics import mean_squared_error

train_mae = mean_absolute_error(y_train, numeric_train_pred)
valid_mae = mean_absolute_error(y_valid, numeric_valid_pred)
valid_rmse = np.sqrt(mean_squared_error(y_valid, numeric_valid_pred))
print(f'Train MAE: {train_mae:.1f} евро')
print(f'Valid MAE: {valid_mae:.1f} евро')
print(f'Valid RMSE: {valid_rmse:.1f} евро')


Как читать очень грубо.

- Train маленький, valid огромный — модель могла зазубрить train.
- Оба большие — модель слабая или данные шумные.
- Близкие ошибки ещё ничего не доказывают «навсегда»: это одна валидация.


### Сведём три варианта в таблицу

Одна линейка — MAE на той же валидации.


In [ ]:
comparison_05 = pd.DataFrame({
    'Модель': ['Медиана train', 'Линейная: Year', 'Линейная: четыре признака'],
    'Valid MAE, евро': [baseline_valid_mae, year_valid_mae, valid_mae]
})
display(comparison_05.round(1))


Кто меньше? На сколько относительно медианы? Для MAE меньше — лучше. Четыре признака не обязаны выиграть: если не выиграли, так и пишем.


### Известные цены и прогнозы вместе

`Error` со знаком: плюс — модель дороже правды. `Abs_error` — величина без знака.


In [ ]:
valid_predictions = X_valid.copy()
valid_predictions['Price'] = y_valid
valid_predictions['Prediction'] = numeric_valid_pred
valid_predictions['Error'] = valid_predictions['Prediction'] - valid_predictions['Price']
valid_predictions['Abs_error'] = valid_predictions['Error'].abs()
display(valid_predictions.head(10))


**Мини-задание.** Выведите пять наибольших `Abs_error`. Большая ошибка модели ещё не доказывает, что цена в исходной строке неверна.


In [ ]:
# Выводим 5 наибольших абсолютных ошибок
top5_errors = valid_predictions.sort_values(by='Abs_error', ascending=False).head(5)
display(top5_errors[['Year', 'Distance', 'Engine', 'km_year', 'Price', 'Prediction', 'Error', 'Abs_error']])

# Объяснение:
# Большая ошибка модели не доказывает опечатку в цене. Автомобиль может иметь скрытые особенности,
# о которых модель не знает (уникальная комплектация, идеальное состояние или, наоборот, последствия ДТП).


### График «известная цена — прогноз»

Каждая точка — одна валидационная запись. Идеал — пунктир «прогноз = цена». Выше пунктира модель завышает, ниже — занижает.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_valid, numeric_valid_pred, s=10, alpha=0.25)
low = min(y_valid.min(), numeric_valid_pred.min())
high = max(y_valid.max(), numeric_valid_pred.max())
ax.plot([low, high], [low, high], 'r--', label='Совпадение')
ax.set(xlabel='Указанная цена, евро', ylabel='Прогноз, евро',
       title='Четыре численных признака: валидация')
ax.legend()
ax.grid(alpha=.3)
plt.show()


Прямая может уехать в отрицательную цену. Машина за минус евро не продаётся. Это не баг кода, это ограничение линейной формы.

Обрезать прогнозы нулём «тихо» не надо: это уже другая модель, её надо заново мерить на valid.


## 9. Дополнительно: MAPE, R² и score

Три числа, чтобы не путать с MAE.

**MAPE** — средний модуль ошибки в **долях** от цены. Sklearn отдаёт долю, ×100 — проценты. Дешёвая машина с промахом 500 евро даёт большой процент, дорогая с промахом 2000 — маленький. Для нулевых цен ломается. Сравнивать MAPE с MAE «кто лучше» вслепую нельзя: разные линейки.

**R²** — насколько квадраты ошибок меньше, чем если бы мы всегда предсказывали среднюю цену **этой же** проверяемой части. 1 — идеально. 0 — как эта константа-среднее. Ниже нуля — хуже константы. Это не «доля правильно угаданных машин»: у регрессии нет правильно/неправильно по одной штуке.

**`score`** — короткая кнопка у модели. У линейной регрессии это как раз R². У классификатора будет другое. В отчёте пишите «R² = …», не «score хороший».


In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, r2_score

print(f'Valid MAPE: {100 * mean_absolute_percentage_error(y_valid, numeric_valid_pred):.1f}%')
print(f'Valid R²: {r2_score(y_valid, numeric_valid_pred):.3f}')
print(f'LinearRegression.score: {numeric_model.score(X_valid_scaled, y_valid):.3f}')


Последние две строки должны совпасть: это одна метрика двумя вызовами. В `score` снова подаём масштабированные признаки, как при обучении.


## 10. Самостоятельная работа и передача в 06

1. Объясните `X`, `y`, `fit`, `predict` на примере одного автомобиля.
2. Выпишите размеры трёх частей и их назначение.
3. Сравните медиану и обе линейные модели по MAE одной валидации.
4. Проверьте пять больших ошибок: покажите признаки, цену, прогноз и направление ошибки.
5. *Дополнительно:* сравните линейную модель с масштабированием и без него на тех же строках. Это не новый признак, а другая подготовка тех же столбцов.

**Что сдавать:** выполненный ноутбук, таблицы сравнения и валидационных прогнозов; краткий вывод из 5–7 предложений. Тестовые метрики пока не вычисляйте.

**Оценивание, 10 баллов:** постановка и входы — 2; разделение — 2; обучение и предсказание — 2; метрики и график — 2; интерпретация и вывод — 2.

Скелет на всю пару:

создали объект → `fit` на train → `predict` только по признакам → ошибку мерим на valid.


### Сохранение результатов

Три файла: сравнение моделей, прогнозы на valid, список «какая строка в какой кусок попала». Если поменять csv, порядок строк или `random_state` — разделение уже другое.


In [ ]:
comparison_05.to_csv('lab05_model_comparison.csv', index=False)
valid_predictions.rename_axis('dataset_row_id').reset_index().to_csv(
    'lab05_valid_predictions.csv', index=False
)
split_manifest = pd.DataFrame({'dataset_row_id': df.index, 'split': ''})
split_manifest.loc[X_train.index, 'split'] = 'train'
split_manifest.loc[X_valid.index, 'split'] = 'valid'
split_manifest.loc[X_test.index, 'split'] = 'test'
split_manifest.to_csv('lab05_split_manifest.csv', index=False)


Работа 06 повторяет тот же отбор строк при том же файле, его порядке и случайных состояниях.

Документация: [первые шаги scikit-learn](https://scikit-learn.org/stable/getting_started.html), [ошибки подготовки данных](https://scikit-learn.org/stable/common_pitfalls.html), [метрики регрессии](https://scikit-learn.org/stable/modules/model_evaluation.html).


## Ответы на самостоятельную работу и итоговый вывод

### 1. Объяснение X, y, fit, predict на примере одного автомобиля
- **X (признаки):** Набор численных параметров авто (например, `[2018 год, 95 000 км пробег, 2000 см³ объем, 15 833 км/год]`). Это входящие данные без указания стоимости.
- **y (целевая переменная):** Известная стоимость данного авто из объявления (например, `14 500` евро).
- **fit (обучение):** Процесс, при котором алгоритм просматривает множество обучающих примеров $(X, y)$ и подбирает оптимизированные веса $w$ для каждого признака и сдвиг $b$.
- **predict (предсказание):** Применение найденного правила к новому вектору $X$ для получения вероятной стоимости $\hat{y}$.

### 2. Размеры выборок и их назначение
- **`train` (60%):** Используется для непосредственной настройки коэффициентов модели и параметров масштабирования (`StandardScaler`).
- **`valid` (20%):** Используется для промежуточного контроля качества, сравнения моделей и выбора оптимальной архитектуры.
- **`test` (20%):** Отложенный финальный экзамен, применяемый строго один раз в конце исследования.

### 3. Сравнение моделей по MAE на валидационной выборке
- **DummyRegressor (медиана):** ~5944.0 евро
- **LinearRegression (1 признак — Year):** ~5133.0 евро
- **LinearRegression (4 признака):** ~1215.6 евро

### 4. Анализ 5 наибольших ошибок
Наибольшие отклонения возникают на редких премиальных авто (где линейная модель сильно занижает стоимость) или на машинах с нестандартным состоянием/комплектацией. Положительная `Error` означает переоценку стоимости моделью, отрицательная — недооценку.

### 5. Сравнение модели с масштабированием и без него
Для классической линейной регрессии (OLS) без регуляризации значение **Valid MAE одинаково с масштабированием и без него** (~1215.6 евро), так как математически веса $w$ компенсируют разницу в масштабах признаков. Однако стандартизация необходима для интерпретируемости коэффициентов и работы сложных алгоритмов (K-NN, Lasso/Ridge).

### Краткий вывод (5–7 предложений)
1. В ходе работы построена модель линейной регрессии для предсказания стоимости автомобилей на основе 4 численных признаков.
2. Данные успешно разделены на три изолированные выборки: `train` (60%), `valid` (20%) и `test` (20%).
3. Базовый ориентир (медиана `train`) показал Valid MAE ~5944 евро, а использование одного признака `Year` снизило ошибку до ~5133 евро.
4. Включение всех четырех признаков позволило значительно улучшить результат, снизив среднюю абсолютную ошибку до ~1215 евро.
5. Масштабирование признаков с помощью `StandardScaler` выполнено строго на обучающей выборке во избежание утечки данных.
6. Проведенный анализ показал ограничение линейной модели при оценке аномальных и редких цен, что обосновывает необходимость перехода к более сложным нелинейным моделям в следующих работах.
